# Data Analysis: Student Performance
*Generated by Autonomous Data Analyst Agent*

**Date:** 2026-07-19 13:53

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('/Users/shashwatsharma/Desktop/autonomous-data-analyst/data/student_data.csv')
print(df.shape)

## Descriptive Statistics Overview
*Calculate central tendency and dispersion for score, study hours, and attendance to establish a baseline for student performance metrics.*

In [ ]:
CHART_PATH = 'chart_task_001.png'

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = ['score', 'study_hours_per_day', 'attendance_percentage']

stats = df[numeric_cols].describe().transpose()
print("--- Descriptive Statistics Summary ---")
print(stats[['mean', 'std', 'min', '50%', 'max']])

correlation_matrix = df[numeric_cols].corr()
print("\n--- Correlation Matrix ---")
print(correlation_matrix)

pass_rate = df['passed'].mean() * 100
print(f"\nOverall Pass Rate: {pass_rate:.2f}%")

top_subject = df.groupby('subject')['score'].mean().sort_values(ascending=False).head(1)
print(f"\nTop performing subject by average score: {top_subject.index[0]} ({top_subject.values[0]:.2f})")

plt.figure(figsize=(12, 6))
sns.boxplot(data=df[numeric_cols])
plt.title('Distribution of Student Performance Metrics')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

**Insight:** The analysis of 2,500 student records reveals an average score of 77.9% with a strong positive correlation of 0.73 between daily study hours and academic performance. While attendance averages 79.8%, it shows a weak correlation of 0.21 with scores, suggesting that study time is a more significant driver of success than physical presence. Notably, all students in the dataset achieved a passing grade, with History emerging as the top-performing subject at an average of 85.34%.

*Confidence: 95%*

## Correlation Analysis of Performance Drivers
*Examine the relationship between study hours, attendance percentage, and final scores to identify which factors have the strongest impact on academic success.*

In [ ]:
CHART_PATH = 'chart_task_002.png'

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

try:
    numeric_cols = ['score', 'study_hours_per_day', 'attendance_percentage']
    corr_matrix = df[numeric_cols].corr()

    print("--- Correlation Analysis: Performance Drivers ---")
    print(f"Correlation between Study Hours and Score: {corr_matrix.loc['study_hours_per_day', 'score']:.4f}")
    print(f"Correlation between Attendance and Score: {corr_matrix.loc['attendance_percentage', 'score']:.4f}")
    print(f"Correlation between Study Hours and Attendance: {corr_matrix.loc['study_hours_per_day', 'attendance_percentage']:.4f}")

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
    plt.title('Correlation Heatmap: Performance Drivers')
    plt.savefig(CHART_PATH, bbox_inches='tight')
    plt.close('all')

    avg_score_by_grade = df.groupby('grade')['score'].mean().sort_values(ascending=False)
    print("\n--- Average Score by Grade ---")
    print(avg_score_by_grade.to_string())

    pass_rate = df.groupby('subject')['passed'].mean() * 100
    print("\n--- Pass Rate by Subject (%) ---")
    print(pass_rate.to_string())

except Exception as e:
    print(f"An error occurred during analysis: {e}")

**Insight:** The analysis reveals a strong positive correlation of 0.735 between daily study hours and student scores, indicating that study time is the primary driver of academic performance. In contrast, attendance shows a weak correlation of 0.206 with scores, and there is virtually no relationship between study hours and attendance. Additionally, pass rates are consistently at 100% across all subjects, suggesting the current curriculum difficulty may be low relative to student capability.

*Confidence: 95%*

## Comparative Performance by Subject
*Group data by 'subject' to compare average scores and pass rates, revealing which subjects are most challenging for the student body.*

In [ ]:
CHART_PATH = 'chart_task_003.png'

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    subject_analysis = df.groupby('subject').agg({
        'score': 'mean',
        'passed': 'mean'
    }).rename(columns={'score': 'avg_score', 'passed': 'pass_rate'})

    subject_analysis = subject_analysis.sort_values(by='avg_score', ascending=False)

    print("--- Comparative Performance by Subject ---")
    print(subject_analysis)
    
    hardest_subject = subject_analysis['avg_score'].idxmin()
    easiest_subject = subject_analysis['avg_score'].idxmax()
    
    print(f"\nMost challenging subject (lowest avg score): {hardest_subject} ({subject_analysis.loc[hardest_subject, 'avg_score']:.2f})")
    print(f"Easiest subject (highest avg score): {easiest_subject} ({subject_analysis.loc[easiest_subject, 'avg_score']:.2f})")

    plt.figure(figsize=(10, 6))
    sns.barplot(x=subject_analysis.index, y=subject_analysis['avg_score'], palette='viridis')
    plt.title('Average Score by Subject')
    plt.ylabel('Average Score')
    plt.xlabel('Subject')
    plt.xticks(rotation=45)
    plt.savefig(CHART_PATH, bbox_inches='tight')
    plt.close('all')

except Exception as e:
    print(f"An error occurred during analysis: {e}")

**Insight:** The analysis of 2,500 student records shows a 100% pass rate across all subjects, indicating that current curriculum standards are highly achievable. However, performance varies significantly by subject, with History yielding the highest average score of 85.34 and Math proving the most challenging with an average score of 70.65. This represents a 14.69-point performance gap between the top and bottom-performing subjects.

*Confidence: 95%*

## Outlier Detection in Student Scores
*Identify students with unusually low or high scores relative to their study hours and attendance to flag potential data entry errors or exceptional cases.*

In [ ]:
CHART_PATH = 'chart_task_004.png'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    df_clean = df.dropna(subset=['score', 'study_hours_per_day', 'attendance_percentage']).copy()
    
    df_clean['score_zscore'] = (df_clean['score'] - df_clean['score'].mean()) / df_clean['score'].std()
    
    df_clean['expected_score'] = (
        df_clean['study_hours_per_day'] * 5 + 
        df_clean['attendance_percentage'] * 0.4
    )
    df_clean['residual'] = df_clean['score'] - df_clean['expected_score']
    
    threshold = 2.5
    outliers = df_clean[
        (df_clean['residual'].abs() > df_clean['residual'].std() * threshold)
    ]
    
    print(f"Total records analyzed: {len(df_clean)}")
    print(f"Outliers detected based on residual analysis: {len(outliers)}")
    print(f"Mean score of outliers: {outliers['score'].mean():.2f}")
    print(f"Top 5 most anomalous students (by residual):")
    print(outliers[['student_id', 'subject', 'score', 'residual']].sort_values('residual', ascending=False).head(5))

    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_clean, x='study_hours_per_day', y='score', alpha=0.5, label='Normal')
    sns.scatterplot(data=outliers, x='study_hours_per_day', y='score', color='red', label='Outliers')
    plt.title('Student Score vs Study Hours (Outliers Highlighted)')
    plt.xlabel('Study Hours per Day')
    plt.ylabel('Score')
    plt.savefig(CHART_PATH, bbox_inches='tight')
    plt.close('all')

except Exception as e:
    print(f"Error during outlier analysis: {e}")

**Insight:** The analysis identified 1,320 outliers out of 2,462 records, representing a high anomaly rate of 53.6%. These students achieved a high mean score of 83.57 despite study hours and attendance metrics that statistically suggest much lower performance. The most significant anomalies are concentrated in the History subject, where students are scoring up to 50 points higher than their input variables would predict.

*Confidence: 75%*

## Distribution of Grade Letters
*Analyze the frequency distribution of 'grade_letter' to understand the overall grading curve and identify the proportion of students in each performance tier.*

In [ ]:
CHART_PATH = 'chart_task_005.png'

import matplotlib.pyplot as plt
import pandas as pd

try:
    grade_counts = df['grade_letter'].value_counts().sort_index()
    grade_pct = df['grade_letter'].value_counts(normalize=True).sort_index() * 100

    print("--- Grade Letter Distribution Summary ---")
    for grade, count in grade_counts.items():
        print(f"Grade {grade}: {count} students ({grade_pct[grade]:.2f}%)")

    plt.figure(figsize=(10, 6))
    grade_counts.plot(kind='bar', color='skyblue', edgecolor='black')
    plt.title('Distribution of Student Grade Letters')
    plt.xlabel('Grade Letter')
    plt.ylabel('Number of Students')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.savefig(CHART_PATH, bbox_inches='tight')
    plt.close('all')

except Exception as e:
    print(f"Error during analysis: {e}")

**Insight:** The analysis of 2,500 students reveals a heavily right-skewed grading distribution, with 81.96% of the student body achieving either an A (44.32%) or a B (37.64%). Only 1.24% of students received a D, indicating that the vast majority of the population is performing at a high level.

*Confidence: 95%*